# Gradient Methods — Interactive Comparison Dashboard

Compare **Fixed-Step Gradient Descent** and **Steepest Descent**
on identical quadratic optimization problems.


In [ ]:
from pathlib import Path
import sys

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

root = Path.cwd()
if not (root / "gradient_methods").is_dir():
    for parent in root.parents:
        if (parent / "gradient_methods").is_dir():
            root = parent
            break

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from gradient_methods.fixed_step_gradient_solver.FixedStepGradient import (
    solve_fixed_step_gradient,
)
from gradient_methods.steepest_descent_solver.SteepestDescent import (
    solve_steepest_descent,
)




In [ ]:

def compare_gradient_methods(
    objective=1,
    dimension=10,
    rho=0.25,
    tolerance=1e-10,
    max_iter=150,
):
    """Compare both solvers on the same quadratic optimization problem."""

    if objective not in (1, 2):
        raise ValueError("objective must be 1 or 2.")
    if not isinstance(dimension, (int, np.integer)) or dimension < 1:
        raise ValueError("dimension must be a positive integer.")

    target = (
        np.ones(dimension)
        if objective == 1
        else np.arange(1, dimension + 1, dtype=float)
    )

    x0 = np.zeros(dimension)
    A = 2.0 * np.eye(dimension)
    b = 2.0 * target

    fixed = solve_fixed_step_gradient(
        u0=x0,
        rho=rho,
        max_iter=max_iter,
        tol=tolerance,
        objective=objective,
    )

    steepest = solve_steepest_descent(
        A,
        b,
        x0=x0,
        rtol=0.0,
        atol=tolerance,
        max_iter=max_iter,
        store_history=True,
    )

    steepest_errors = np.linalg.norm(
        steepest.history - target,
        axis=1,
    )

    results = {
        "Fixed-Step Gradient": {
            "solution": fixed.solution,
            "iterations": fixed.iterations,
            "converged": fixed.converged,
            "costs": np.asarray(fixed.cost_history),
            "gradient_norms": np.asarray(fixed.gradient_norms),
            "solution_errors": np.sqrt(
                np.maximum(fixed.cost_history, 0.0)
            ),
            "message": fixed.message,
        },
        "Steepest Descent": {
            "solution": steepest.solution,
            "iterations": steepest.iterations,
            "converged": steepest.converged,
            "costs": steepest_errors**2,
            "gradient_norms": 2.0 * steepest_errors,
            "solution_errors": steepest_errors,
            "message": steepest.message,
        },
    }

    return target, results


In [ ]:

def plot_gradient_comparison(results):
    """Plot convergence histories for both gradient methods."""

    metrics = [
        ("costs", "Objective value", "J(x)"),
        ("gradient_norms", "Gradient norm", "||gradient J(x)||"),
        ("solution_errors", "Solution error", "||x - x*||"),
    ]

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

    for ax, (key, title, ylabel) in zip(axes, metrics):
        for name, result in results.items():
            values = np.asarray(result[key], dtype=float)
            iterations = np.arange(len(values))

            # Display floor only; underlying values remain unchanged.
            safe_values = np.maximum(values, 1e-14)

            ax.semilogy(
                iterations,
                safe_values,
                marker="o",
                markersize=3,
                linewidth=1.8,
                label=name,
            )

        ax.set_title(title)
        ax.set_xlabel("Iteration")
        ax.set_ylabel(ylabel)
        ax.grid(True, which="both", alpha=0.3)
        ax.legend(fontsize=8)

    

    fig.tight_layout()
    return fig

In [ ]:

from IPython.display import HTML
from html import escape

def display_gradient_comparison_table(results):
    """Display numerical results for both gradient methods."""

    rows = []

    for name, result in results.items():
        status = "Converged" if result["converged"] else "Not converged"

        rows.append(
            "<tr>"
            f"<td>{escape(name)}</td>"
            f"<td>{result['iterations']}</td>"
            f"<td>{status}</td>"
            f"<td>{result['costs'][-1]:.3e}</td>"
            f"<td>{result['gradient_norms'][-1]:.3e}</td>"
            f"<td>{result['solution_errors'][-1]:.3e}</td>"
            "</tr>"
        )

    html = """
    <div style="overflow-x:auto; margin:12px 0;">
    <table style="width:100%; border-collapse:collapse; text-align:center;">
      <thead>
        <tr style="background:#17365d; color:white;">
          <th style="padding:12px;">Method</th>
          <th style="padding:12px;">Iterations</th>
          <th style="padding:12px;">Status</th>
          <th style="padding:12px;">Final objective</th>
          <th style="padding:12px;">Gradient norm</th>
          <th style="padding:12px;">Solution error</th>
        </tr>
      </thead>
      <tbody>
    """ + "".join(rows) + """
      </tbody>
    </table>
    </div>
    """

    display(HTML(html))


In [ ]:

# Interactive controls
objective_control = widgets.Dropdown(
    options=[("J1 — Target vector of ones", 1),
             ("J2 — Target vector (1, 2, ..., n)", 2)],
    value=1,
    description="Objective:",
    style={"description_width": "initial"},
)

dimension_control = widgets.IntSlider(
    value=10, min=2, max=100, step=1,
    description="Dimension:",
    continuous_update=False,
    style={"description_width": "initial"},
)

rho_control = widgets.FloatSlider(
    value=0.25, min=0.05, max=1.20, step=0.05,
    description="Fixed step size:",
    readout_format=".2f",
    continuous_update=False,
    style={"description_width": "initial"},
)

tolerance_control = widgets.Dropdown(
    options=[
        ("1e-6", 1e-6),
        ("1e-8", 1e-8),
        ("1e-10", 1e-10),
        ("1e-12", 1e-12),
    ],
    value=1e-10,
    description="Tolerance:",
    style={"description_width": "initial"},
)

iterations_control = widgets.IntSlider(
    value=150, min=10, max=500, step=10,
    description="Max iterations:",
    continuous_update=False,
    style={"description_width": "initial"},
)

run_button = widgets.Button(
    description="Run Comparison",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width="200px", height="40px"),
)

status = widgets.HTML(
    value="<span style='color:#666;'>Choose parameters and run the comparison.</span>"
)

plot_output = widgets.Output()
table_output = widgets.Output()

def run_comparison(_):
    run_button.disabled = True
    status.value = "<span style='color:#2563eb;'>Running comparison...</span>"

    try:
        target, results = compare_gradient_methods(
            objective=objective_control.value,
            dimension=dimension_control.value,
            rho=rho_control.value,
            tolerance=tolerance_control.value,
            max_iter=iterations_control.value,
        )

        with plot_output:
            clear_output(wait=True)
            fig = plot_gradient_comparison(results)
            plt.show()
            plt.close(fig)

        with table_output:
            clear_output(wait=True)
            display_gradient_comparison_table(results)

        status.value = (
            "<span style='color:#15803d;'>"
            "&#10003; Comparison completed successfully."
            "</span>"
        )

    except Exception as exc:
        status.value = (
            "<span style='color:#b91c1c;'>"
            f"Comparison failed: {escape(str(exc))}"
            "</span>"
        )
    finally:
        run_button.disabled = False

run_button.on_click(run_comparison)


# Track changes to the experiment configuration.
def mark_results_outdated(change):
    if change["new"] != change["old"]:
        status.value = (
            "<span style='color:#b45309;'>"
            "&#9888; Parameters changed — click "
            "<b>Run Comparison</b> to update the results."
            "</span>"
        )

for control in (
    objective_control,
    dimension_control,
    rho_control,
    tolerance_control,
    iterations_control,
):
    control.observe(mark_results_outdated, names="value")




# Compact mathematical explanation
math_box = widgets.HTML(
    value="""
    <div style="
        border:1px solid #dbe3ef;
        border-radius:6px;
        padding:12px 18px;
        background:#f8fafc;
        font-size:13px;
        line-height:1.65;
    ">
      <b style="font-size:15px;">Mathematical Formulation</b>

      <p style="margin:7px 0;">
        <b>Objective:</b>
        J(x) = ||x - x*||²
      </p>

      <p style="margin:5px 0;">
        <b>J1:</b> x* = (1, ..., 1)<br>
        <b>J2:</b> x* = (1, 2, ..., n)
      </p>

      <p style="margin:5px 0;">
        <b>Gradient:</b> ∇J(x) = 2(x - x*)<br>
        <b>Hessian:</b> ∇²J(x) = 2I
      </p>

      <p style="margin:5px 0;">
        <b>Fixed-Step:</b> xₖ₊₁ = xₖ - ρ∇J(xₖ)
      </p>

      <p style="margin:5px 0;">
        <b>Steepest Descent:</b>
        exact line search along -∇J(xₖ)
      </p>

      <p style="margin:7px 0 0;color:#475569;">
        Both methods use the same initial point and stopping tolerance.
        For these objectives, exact-line-search Steepest Descent
        converges in one iteration.
      </p>
    </div>
    """
)

controls = widgets.VBox(
    [
        widgets.HTML("<b>Experiment Configuration</b>"),
        objective_control,
        dimension_control,
        rho_control,
        tolerance_control,
        iterations_control,
        run_button,
        status,
    ],
    layout=widgets.Layout(
        width="34%",
        padding="10px",
        border="1px solid #dbe3ef",
    ),
)

math_box.layout = widgets.Layout(width="64%")

top_row = widgets.HBox(
    [controls, math_box],
    layout=widgets.Layout(
        width="100%",
        justify_content="space-between",
        align_items="flex-start",
    ),
)

dashboard = widgets.VBox(
    [
        top_row,
        widgets.HTML(
            "<b style='font-size:15px;'>Convergence Comparison</b>"
        ),
        plot_output,
        widgets.HTML(
            "<b style='font-size:15px;'>Numerical Results</b>"
        ),
        table_output,
    ],
    layout=widgets.Layout(
        width="100%",
        gap="5px",
    ),
)

display(dashboard)
